In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Tests de Calidad de Datos (Data Quality)
# MAGIC **Objetivo:** Validar que los datos de la capa Silver sean confiables antes de dar por finalizado el pipeline.
from pyspark.sql.functions import col, count, when

print("⏳ Iniciando los tests de calidad...", flush=True)

# 1. Leer los datos de hoy de la capa Silver
df_silver = spark.sql("SELECT * FROM silver_coingecko_market WHERE ingestion_date = CURRENT_DATE()")

# --- TEST 1: ¿Trajimos datos? ---
total_filas = df_silver.count()
if total_filas == 0:
    raise Exception("❌ TEST FALLIDO: La tabla Silver está vacía para la fecha de hoy. La API pudo haber fallado.")
print(f"✅ TEST 1 PASADO: Se encontraron {total_filas} registros para hoy.",  flush=True)

# --- TEST 2: ¿Hay precios nulos? ---
nulos_precio = df_silver.filter(col("current_price_usd").isNull()).count()
if nulos_precio > 0:
    raise Exception(f"❌ TEST FALLIDO: Hay {nulos_precio} monedas con precio nulo (NULL).")
print("✅ TEST 2 PASADO: No hay precios nulos.", flush=True)

# --- TEST 3: ¿Hay precios negativos o en cero? ---
precios_invalidos = df_silver.filter(col("current_price_usd") <= 0).count()
if precios_invalidos > 0:
    raise Exception(f"❌ TEST FALLIDO: Hay {precios_invalidos} monedas con precio menor o igual a 0.")
print("✅ TEST 3 PASADO: Todos los precios son mayores a 0.", flush=True)

# --- TEST 4: ¿Hay monedas duplicadas? ---
total_distintos = df_silver.select("id").distinct().count()
if total_filas != total_distintos:
    raise Exception(f"❌ TEST FALLIDO: Hay monedas duplicadas. Filas totales: {total_filas}, monedas únicas: {total_distintos}.")
print("✅ TEST 4 PASADO: No hay monedas duplicadas.", flush=True)

# Si llegamos hasta acá, todo está perfecto
print("\n🎉 ¡TODOS LOS TESTS PASARON! Los datos están listos y son confiables.", flush=True)